In [0]:
from pyspark.sql import functions as F
from datetime import datetime

start_time = datetime.now()

def audit_log(run_id, schema_name, table_name, start_time, status, row_count, error):
    end_time = datetime.now()
    log = {
        'run_id': run_id,
        'schema_name': schema_name,
        'table_name': table_name,
        'start_time': str(start_time),
        'end_time': str(end_time),
        'duration': (end_time - start_time).total_seconds(),
        'status': status,
        'rows_written': int(row_count or 0),
        'error_message': error
    }
    schema = ("run_id string, schema_name string, table_name string, "
              "start_time string, end_time string, duration double, "
              "status string, rows_written bigint, error_message string")

    spark.createDataFrame([log], schema) \
         .write.mode('append').saveAsTable('retail.ops.audit_logs')

dbutils.widgets.text('catalog', 'retail', 'Catalog')
dbutils.widgets.text('schema', 'gold', 'Schema')
dbutils.widgets.text('table', 'daily_sales', 'Table')
dbutils.widgets.text('run_id', 'test', 'Run ID')

catalog = dbutils.widgets.get('catalog')
schema = dbutils.widgets.get('schema')
table = dbutils.widgets.get('table')
run_id = dbutils.widgets.get('run_id')

In [0]:
STATUS = None
error_message = None
row_count = 0

try:

    fact_orders = spark.table(f'{catalog}.{schema}.fact_orders')
    dim_customer = spark.table(f'{catalog}.{schema}.dim_customer')

    daily_sales = (
        fact_orders.alias('T1')
        .join(
            dim_customer.alias('T2'),
            on = F.col('T1.customer_id')==F.col('T2.customer_id'),
            how = 'inner'
        )
        .groupBy(
            F.col('T1.transaction_date'),
            F.col('T2.city'),
            F.col('T2.segment')
        )
        .agg(
            F.sum(F.col('T1.amount')).alias('total_sales')
        )
        .orderBy(
            F.col('T1.transaction_date').asc()
        )
    )

    row_count = daily_sales.count()

    (
        daily_sales
        .write
        .mode('overwrite')
        .saveAsTable(f'{catalog}.{schema}.{table}')
    )
    STATUS = 'SUCCESS'

except Exception as e:
    error_message = str(e)[:1000]
    STATUS = 'FAILED'

finally: 
    audit_log(
        run_id = run_id,
        schema_name = schema,
        table_name = table,
        start_time = start_time,
        status = STATUS,
        row_count = row_count,
        error = error_message
    )